# ml_C_trees_codes.ipynb

**所属章节**：Chapter C 树模型  
**用途**：生成讲义中的全部配图（9张），保存至 `figs/` 目录  
**运行说明**：顺序执行，约 3–5 分钟（SHAP 计算耗时较长）


In [4]:
# !pip install shap

In [5]:
# Global setup
import os
os.environ.setdefault('MPLCONFIGDIR', os.path.join(os.getcwd(), '.mplconfig'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import make_regression, make_classification
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import train_test_split
try:
    import shap
except ImportError:
    shap = None
import warnings
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

# Chinese font support
from matplotlib import font_manager

candidate_fonts = [
    'Microsoft YaHei', 'SimHei', 'SimSun', 'Arial Unicode MS',
    'Noto Sans CJK SC', 'Source Han Sans SC', 'WenQuanYi Micro Hei'
]
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
zh_fonts = [f for f in candidate_fonts if f in available_fonts]
ZH_FONT = zh_fonts[0] if zh_fonts else None
if zh_fonts:
    plt.rcParams['font.sans-serif'] = zh_fonts + plt.rcParams['font.sans-serif']
else:
    print('Warning: no common Chinese font found; Chinese text may render as boxes.')
FP = font_manager.FontProperties(family=ZH_FONT) if ZH_FONT else font_manager.FontProperties()
FPB = font_manager.FontProperties(family=ZH_FONT, weight='bold') if ZH_FONT else font_manager.FontProperties(weight='bold')
plt.rcParams['axes.unicode_minus'] = False

C = {'primary':'#0B3D91','secondary':'#B8860B','tertiary':'#2F5E9E',
     'neutral':'#878787','highlight':'#6B4E00','fill':'#D6E2F3'}
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,
    'font.size':11,'axes.titlesize':13,'axes.labelsize':11,
    'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'grid.linestyle':'--'})
SEED=42; np.random.seed(SEED)
print('Global setup complete')


全局设置完成


---
## 共用数据集


In [6]:
# 生成回归数据（n=400, p=20, s=8 个真实相关特征）
X, y, coef_true = make_regression(
    n_samples=400, n_features=20, n_informative=8,
    noise=0.5, coef=True, random_state=SEED)
feat_names = [f'X{i+1}' for i in range(20)]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=SEED)
print(f'训练集: {X_tr.shape}, 测试集: {X_te.shape}')

训练集: (300, 20), 测试集: (100, 20)


---
## 图1：决策树结构图


In [9]:
# 训练浅层决策树（深度=3，便于可视化）
dt = DecisionTreeRegressor(max_depth=3, random_state=SEED)
dt.fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(dt, feature_names=feat_names, filled=True, rounded=True,
          ax=ax, fontsize=10,
          node_ids=False, proportion=False)
ax.set_title('决策树结构图（回归树，深度=3）', fontproperties=FPB, fontsize=13)
fig.savefig('./figs/fig_C_decision_tree.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_C_decision_tree.png 已保存')

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\usr\\share\\fonts\\opentype\\noto\\NotoSansCJK-Medium.ttc'

Error in callback <function _draw_all_if_interactive at 0x000001D175A59120> (for post_execute), with arguments args (),kwargs {}:


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\usr\\share\\fonts\\opentype\\noto\\NotoSansCJK-Medium.ttc'

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\usr\\share\\fonts\\opentype\\noto\\NotoSansCJK-Medium.ttc'

<Figure size 1680x720 with 1 Axes>

---
## 图2：Bagging 如何降低方差


In [10]:
# 1D 数据，展示单树 vs 随机森林预测波动
np.random.seed(SEED)
x1d = np.linspace(0, 1, 200).reshape(-1,1)
true_f = lambda x: np.sin(4*np.pi*x).ravel()
n_train_1d = 30; n_rep = 50

preds_tree = np.zeros((n_rep, 200))
preds_rf   = np.zeros((n_rep, 200))
for i in range(n_rep):
    xi = np.random.uniform(0, 1, n_train_1d).reshape(-1,1)
    yi = true_f(xi) + np.random.normal(0, 0.3, n_train_1d)
    preds_tree[i] = DecisionTreeRegressor(random_state=i).fit(xi,yi).predict(x1d)
    preds_rf[i]   = RandomForestRegressor(n_estimators=50,random_state=i).fit(xi,yi).predict(x1d)

fig, axes = plt.subplots(1,2,figsize=(12,4.5))
for ax, preds, title in zip(axes,
    [preds_tree, preds_rf],
    ['单棵决策树（高方差）','随机森林（低方差）']):
    for p in preds[:20]:
        ax.plot(x1d, p, color=C['fill'], lw=0.8, alpha=0.6)
    ax.plot(x1d, true_f(x1d), color=C['highlight'], lw=2, label='真实函数')
    ax.plot(x1d, preds.mean(0), color=C['primary'], lw=2.2, ls='--', label='均值预测')
    ax.set_title(title, fontproperties=FPB, fontsize=12)
    ax.set_xlabel('x', fontproperties=FP)
    ax.set_ylabel('预测值', fontproperties=FP)
    ax.legend(prop=FP)
    var_str = f'预测方差 = {preds.var(axis=0).mean():.3f}'
    ax.text(0.05, 0.92, var_str, transform=ax.transAxes,
            fontproperties=FP, fontsize=10, color=C['secondary'])
fig.suptitle('Bagging 降低方差：50 次重复实验', fontproperties=FPB, fontsize=13)
fig.tight_layout()
fig.savefig('figs/fig_C_bagging_variance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_C_bagging_variance.png 已保存')

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\usr\\share\\fonts\\opentype\\noto\\NotoSansCJK-Regular.ttc'

Error in callback <function _draw_all_if_interactive at 0x000001D175A59120> (for post_execute), with arguments args (),kwargs {}:


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\usr\\share\\fonts\\opentype\\noto\\NotoSansCJK-Regular.ttc'

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\usr\\share\\fonts\\opentype\\noto\\NotoSansCJK-Regular.ttc'

<Figure size 1440x540 with 2 Axes>

---
## 图3：随机森林 OOB 误差收敛


In [ ]:
# 随着树数量增加，OOB 误差的变化
n_trees_list = list(range(5, 305, 10))
oob_errors = []
for n_trees in n_trees_list:
    rf_tmp = RandomForestRegressor(
        n_estimators=n_trees, oob_score=True, random_state=SEED, n_jobs=1)
    rf_tmp.fit(X_tr, y_tr)
    # OOB 误差 = 1 - OOB R2（越小越好）
    oob_errors.append(1 - rf_tmp.oob_score_)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(n_trees_list, oob_errors, color=C['primary'], lw=2.2)
ax.axhline(min(oob_errors), color=C['neutral'], lw=1.2, ls=':', label=f'最低误差 = {min(oob_errors):.4f}')
ax.set_xlabel('树的数量 B', fontproperties=FP, fontsize=11)
ax.set_ylabel('OOB 误差（1 - OOB R²）', fontproperties=FP, fontsize=11)
ax.set_title('随机森林 OOB 误差随树数量的收敛', fontproperties=FPB, fontsize=13)
ax.legend(prop=FP)
fig.tight_layout()
fig.savefig('figs/fig_C_rf_oob_error.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_C_rf_oob_error.png 已保存')

---
## 图4：梯度提升迭代过程


In [ ]:
# 1D 数据，逐步展示 Boosting 迭代过程（4 轮）
np.random.seed(SEED)
n_boost = 40
x_b = np.sort(np.random.uniform(0, 1, n_boost))
y_b = np.sin(4*np.pi*x_b) + np.random.normal(0, 0.2, n_boost)
x_grid = np.linspace(0, 1, 200)

n_stages = [1, 3, 10, 50]  # 使用的树数量
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.subplots_adjust(hspace=0.45, wspace=0.3)

F_current = np.full(n_boost, y_b.mean())  # 初始预测：均值
F_grid    = np.full(200, y_b.mean())

for col, n_t in enumerate(n_stages):
    gbm = GradientBoostingRegressor(
        n_estimators=n_t, max_depth=2, learning_rate=0.3, random_state=SEED)
    gbm.fit(x_b.reshape(-1,1), y_b)
    pred_grid = gbm.predict(x_grid.reshape(-1,1))
    residuals = y_b - gbm.predict(x_b.reshape(-1,1))

    # 上行：当前预测 vs 真实值
    ax_top = axes[0, col]
    ax_top.scatter(x_b, y_b, color=C['neutral'], s=18, alpha=0.7, label='观测值')
    ax_top.plot(x_grid, pred_grid, color=C['primary'], lw=2)
    ax_top.plot(x_grid, np.sin(4*np.pi*x_grid), color=C['highlight'],
                lw=1.5, ls='--', alpha=0.6, label='真实函数')
    ax_top.set_title(f'第{n_t}轮预测', fontproperties=FPB, fontsize=10)
    mse_val = np.mean(residuals**2)
    ax_top.text(0.05, 0.92, f'MSE={mse_val:.3f}',
               transform=ax_top.transAxes, fontproperties=FP, fontsize=10,
               color=C['secondary'])

    # 下行：残差
    ax_bot = axes[1, col]
    ax_bot.scatter(x_b, residuals, color=C['secondary'], s=18, alpha=0.8)
    ax_bot.axhline(0, color='black', lw=0.8, alpha=0.5)
    ax_bot.set_title(f'残差（待修正）', fontproperties=FPB, fontsize=10)
    ax_bot.set_xlabel('x', fontproperties=FP, fontsize=10)

for ax in axes.flat:
    ax.set_xlim(-0.02, 1.02)
    ax.tick_params(labelsize=10)

axes[0,0].legend(prop=FP, fontsize=10)
fig.suptitle('梯度提升迭代过程：预测逐步逼近真实值，残差逐步减小',
             fontproperties=FPB, fontsize=12)
fig.savefig('figs/fig_C_boosting_steps.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_C_boosting_steps.png 已保存')

---
## 图5：MDI vs 置换重要性对比


In [ ]:
# 训练随机森林，计算两种特征重要性
rf_main = RandomForestRegressor(n_estimators=200, oob_score=True,
                                random_state=SEED, n_jobs=1)
rf_main.fit(X_tr, y_tr)

# MDI 特征重要性（来自训练过程，快）
mdi_imp = rf_main.feature_importances_

# 置换重要性（基于测试集，更可靠）
perm_result = permutation_importance(
    rf_main, X_te, y_te, n_repeats=20, random_state=SEED, n_jobs=1)
perm_imp  = perm_result.importances_mean
perm_std  = perm_result.importances_std

# 按 MDI 排序，取前 15
top15_idx = np.argsort(mdi_imp)[::-1][:15]
top15_names = [feat_names[i] for i in top15_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
fig.subplots_adjust(wspace=0.4, top=0.78)

# 左图：MDI
ax = axes[0]
vals = mdi_imp[top15_idx]
ax.barh(range(15), vals[::-1], color=C['primary'], alpha=0.8)
ax.set_yticks(range(15))
ax.set_yticklabels([top15_names[14-i] for i in range(15)], fontproperties=FP, fontsize=10)
ax.set_xlabel('MDI 特征重要性', fontproperties=FP, fontsize=11)
ax.set_title('均值不纯度下降（MDI）\n训练集计算，速度快但可能有偏', fontproperties=FPB, fontsize=11)

# 右图：置换重要性（含误差带）
ax2 = axes[1]
perm_vals = perm_imp[top15_idx]
perm_stds = perm_std[top15_idx]
ax2.barh(range(15), perm_vals[::-1], xerr=perm_stds[::-1],
         color=C['secondary'], alpha=0.8, capsize=3)
ax2.set_yticks(range(15))
ax2.set_yticklabels([top15_names[14-i] for i in range(15)], fontproperties=FP, fontsize=10)
ax2.set_xlabel('置换重要性（均值 ± 标准差）', fontproperties=FP, fontsize=11)
ax2.set_title('置换重要性\n测试集计算，更稳健（含误差带）', fontproperties=FPB, fontsize=11)
ax2.axvline(0, color='black', lw=0.8, alpha=0.5)

fig.suptitle('特征重要性：MDI vs 置换重要性对比（前15个特征）',
             fontproperties=FPB, fontsize=13, y=0.96)
fig.savefig('figs/fig_C_feature_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_C_feature_importance.png 已保存')

---
## 图6 & 图7：SHAP beeswarm 图与 waterfall 图


In [ ]:
if shap is None:
    print('Skip SHAP figures: shap is not installed in this environment.')
else:
    # 计算 SHAP 值（TreeExplainer 对树模型极快）
    explainer   = shap.TreeExplainer(rf_main)
    shap_values = explainer.shap_values(X_te)

    from matplotlib.colors import LinearSegmentedColormap
    shap_cmap = LinearSegmentedColormap.from_list('deep_blue_gold', [C['primary'], C['secondary']])
    shap.plots.colors.blue_rgb = np.array([0x0B/255, 0x3D/255, 0x91/255])
    shap.plots.colors.light_blue_rgb = np.array([0xD6/255, 0xE2/255, 0xF3/255])
    shap.plots.colors.red_rgb = np.array([0xB8/255, 0x86/255, 0x0B/255])
    shap.plots.colors.light_red_rgb = np.array([0xF2/255, 0xD2/255, 0x7A/255])

    # 图6：beeswarm 图（全局解释，前10个特征）
    plt.figure(figsize=(9, 6))
    shap.summary_plot(
        shap_values, X_te,
        feature_names=feat_names,
        max_display=10,
        cmap=shap_cmap,
        show=False
    )
    plt.title('SHAP Beeswarm 图（前10个重要特征）',
              fontproperties=FPB, fontsize=12)
    plt.tight_layout()
    plt.savefig('figs/fig_C_shap_beeswarm.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print('fig_C_shap_beeswarm.png 已保存')

    # 图7：waterfall 图（单样本局部解释）
    shap_exp = shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_te[0],
        feature_names=feat_names
    )
    plt.figure(figsize=(9, 5))
    shap.plots.waterfall(shap_exp, max_display=12, show=False)
    plt.title('SHAP Waterfall 图（测试集第1个样本）',
              fontproperties=FPB, fontsize=12)
    plt.tight_layout()
    plt.savefig('figs/fig_C_shap_waterfall.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    print('fig_C_shap_waterfall.png 已保存')

---
## 图8：部分依赖图（PDP + ICE）


In [ ]:
# 对最重要的2个特征绘制 PDP + ICE
top2 = np.argsort(mdi_imp)[::-1][:2].tolist()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for i, feat_idx in enumerate(top2):
    ax = axes[i]
    # ICE 曲线（细线，个体样本）
    from sklearn.inspection import PartialDependenceDisplay
    disp = PartialDependenceDisplay.from_estimator(
        rf_main, X_te, [feat_idx],
        kind='both',           # 同时显示 PDP 和 ICE
        subsample=50,          # ICE 只取50个样本（避免过于密集）
        random_state=SEED,
        ax=ax,
        ice_lines_kw={'color': C['fill'], 'alpha': 0.4},
        pd_line_kw={'color': C['primary'], 'lw': 2.5, 'label': 'PDP（均值）'}
    )
    ax.set_title(f'PDP + ICE：{feat_names[feat_idx]}',
                 fontproperties=FPB, fontsize=11)
    ax.set_xlabel(f'{feat_names[feat_idx]}', fontproperties=FP, fontsize=10)
    ax.set_ylabel('预测值', fontproperties=FP, fontsize=10)
    ax.legend(prop=FP, fontsize=10)

fig.suptitle('部分依赖图（PDP，粗线）与个体条件期望图（ICE，细线）',
             fontproperties=FPB, fontsize=12)
fig.tight_layout()
fig.savefig('figs/fig_C_pdp.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_C_pdp.png 已保存')

---
## 图9：线性模型 vs 树模型在非线性数据上的对比


In [ ]:
# 生成非线性数据：y = sin(x1) * x2^2 + noise
np.random.seed(SEED)
n_nl = 300
x1 = np.random.uniform(-3, 3, n_nl)
x2 = np.random.uniform(0, 3, n_nl)
y_nl = np.sin(x1) * x2**2 + np.random.normal(0, 0.5, n_nl)
X_nl = np.column_stack([x1, x2,
                         np.random.normal(0,1,(n_nl,8))])  # 8个噪声特征
X_tr_nl, X_te_nl, y_tr_nl, y_te_nl = train_test_split(
    X_nl, y_nl, test_size=0.3, random_state=SEED)

# 拟合两种模型
lr_nl = LinearRegression().fit(X_tr_nl, y_tr_nl)
rf_nl = RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=1).fit(X_tr_nl, y_tr_nl)

r2_lr = lr_nl.score(X_te_nl, y_te_nl)
r2_rf = rf_nl.score(X_te_nl, y_te_nl)

# 绘图：预测值 vs 真实值
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, preds, title, r2, col in zip(
    axes,
    [lr_nl.predict(X_te_nl), rf_nl.predict(X_te_nl)],
    ['线性回归', '随机森林'],
    [r2_lr, r2_rf],
    [C['tertiary'], C['primary']]
):
    ax.scatter(y_te_nl, preds, color=col, alpha=0.5, s=20)
    lim = max(abs(y_te_nl).max(), abs(preds).max()) * 1.05
    ax.plot([-lim,lim],[-lim,lim],'k--',lw=1,alpha=0.4)
    ax.set_xlabel('真实值', fontproperties=FP, fontsize=11)
    ax.set_ylabel('预测值', fontproperties=FP, fontsize=11)
    ax.set_title(f'{title}\n测试集 R² = {r2:.4f}',
                 fontproperties=FPB, fontsize=12)

fig.suptitle('非线性数据（y = sin(x1)*x2² + noise）：线性模型 vs 随机森林',
             fontproperties=FPB, fontsize=12)
fig.tight_layout()
fig.savefig('figs/fig_C_linear_vs_tree.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('fig_C_linear_vs_tree.png 已保存')

In [ ]:
print('\n' + '='*55)
print('Chapter C codes.ipynb 所有图形生成完成')
figs = [f for f in sorted(os.listdir('figs')) if f.startswith('fig_C')]
for f in figs:
    print(f'  {f}  ({os.path.getsize("figs/"+f)//1024} KB)')